<a href="https://colab.research.google.com/github/jasonjavier1/Prompt-Engineering1/blob/main/QUESTION_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Prompt 1: Generate Python code that takes in text as input and summarizes it in 3–5 sentences.
# Make sure it is concise, easy to understand, and written for a general audience. Allow me to
# paste a text and run the code.

#Prompt 2: Now review the code you created and improve it. Critique your work based on these criteria:
# Accuracy: Ensure all key facts are correct. Clarity: Make the summary easy to read and understand.
# Audience: Appropriate for a general audience with no prior knowledge of this topic.
# Length Limits: Keep it concise (3-5 sentences) for each topic within. Avoid bullet points and
# keep sentences in simple terms. Formatting: Use complete sentences and clear paragraph structure.
# Provide a short structured evaluation under each criterion. Produce a fully revised summary that
# corrects all identified issues. Ensure the revised version clearly improves clarity and structure
# compared to the original.


In [8]:
import re
import heapq
from collections import Counter

def summarize_text(text, min_sentences=3, max_sentences=5):
    """
    Summarizes input text into 3–5 sentences using frequency-based scoring.
    """

    if not text or not text.strip():
        return "Error: No text provided. Please paste valid text."

    # Clean text
    text = text.strip()

    # Split into sentences
    sentences = re.split(r'(?<=[.!?]) +', text)

    if len(sentences) < min_sentences:
        return "Error: Text is too short to summarize. Please provide a longer paragraph."

    # Remove punctuation and lowercase for word analysis
    words = re.findall(r'\b\w+\b', text.lower())

    if not words:
        return "Error: Text does not contain valid words."

    # Remove common stopwords (basic list)
    stopwords = {
        "the", "is", "in", "and", "to", "of", "a", "that", "it", "on",
        "for", "as", "with", "its", "are", "was", "by", "an", "be",
        "this", "which", "or", "from", "at", "but", "not", "have",
        "has", "had", "they", "their", "them", "can", "will", "would"
    }

    filtered_words = [word for word in words if word not in stopwords]

    if not filtered_words:
        return "Error: Not enough meaningful words to summarize."

    # Word frequency
    word_freq = Counter(filtered_words)

    # Score sentences
    sentence_scores = {}
    for sentence in sentences:
        sentence_word_count = len(re.findall(r'\b\w+\b', sentence))
        if sentence_word_count < 4:  # Skip very short sentences
            continue

        score = 0
        for word in re.findall(r'\b\w+\b', sentence.lower()):
            if word in word_freq:
                score += word_freq[word]

        sentence_scores[sentence] = score

    if not sentence_scores:
        return "Error: Unable to generate summary from provided text."

    # Determine number of sentences for summary
    summary_length = min(max_sentences, max(min_sentences, len(sentences) // 3))

    # Select top sentences
    best_sentences = heapq.nlargest(summary_length, sentence_scores, key=sentence_scores.get)

    # Preserve original order
    best_sentences.sort(key=lambda s: sentences.index(s))

    summary = " ".join(best_sentences)

    return summary


# --------- USER INPUT ---------
if __name__ == "__main__":
    print("Paste your text below. Press Enter twice when finished:\n")

    user_input = ""
    while True:
        line = input()
        if line == "":
            break
        user_input += line + " "

    result = summarize_text(user_input)

    print("\n--- SUMMARY ---\n")
    print(result)

Paste your text below. Press Enter twice when finished:

What are AI Agents? AI agents are autonomous or semi-autonomous software systems that leverage Large Language Models (LLMs) and other AI tools to perceive information, make decisions, and perform tasks without continuous human input. Unlike traditional software programs, which operate based on pre-coded rules, AI agents can dynamically adapt their behavior based on real-time data and user intent. They can integrate APIs, access tools, generate responses, and even coordinate multi-step processes. Example Use Cases: • A personal assistant AI that manages your schedule, books meetings, and coordinates travel. • A customer service AI agent that handles inquiries, accesses internal databases, and resolves issues proactively. • An autonomous researcher that gathers sources, summarizes content, and presents reports. Defining Prompt Engineering Prompt Engineering is the art and science of crafting instructions that guide LLMs (like GPT-4

In [9]:
import re
import heapq
from collections import Counter

def clean_text(text):
    """Normalize spacing and strip extra whitespace."""
    return re.sub(r'\s+', ' ', text).strip()

def split_sentences(text):
    """
    Improved sentence splitting that handles common punctuation.
    """
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if len(s.strip()) > 0]

def summarize_text(text, min_sentences=3, max_sentences=5):
    """
    Generate a concise 3–5 sentence summary using
    normalized frequency-based scoring and redundancy control.
    """

    if not text or not text.strip():
        return "Error: No text provided. Please enter valid text."

    text = clean_text(text)
    sentences = split_sentences(text)

    if len(sentences) < min_sentences:
        return "Error: Text is too short to summarize. Please provide more content."

    # Extract words
    words = re.findall(r'\b[a-zA-Z]+\b', text.lower())

    if not words:
        return "Error: No valid words found in the text."

    # Basic stopword list
    stopwords = {
        "the", "is", "in", "and", "to", "of", "a", "that", "it", "on",
        "for", "as", "with", "its", "are", "was", "by", "an", "be",
        "this", "which", "or", "from", "at", "but", "not", "have",
        "has", "had", "they", "their", "them", "can", "will", "would",
        "into", "about", "than", "other", "more", "also"
    }

    filtered_words = [word for word in words if word not in stopwords]

    if not filtered_words:
        return "Error: Not enough meaningful content to summarize."

    word_freq = Counter(filtered_words)

    # Normalize frequencies
    max_freq = max(word_freq.values())
    for word in word_freq:
        word_freq[word] /= max_freq

    sentence_scores = {}
    for sentence in sentences:
        sentence_words = re.findall(r'\b[a-zA-Z]+\b', sentence.lower())
        if len(sentence_words) < 5:
            continue

        score = sum(word_freq.get(word, 0) for word in sentence_words)

        # Normalize by sentence length to avoid bias toward long sentences
        score = score / len(sentence_words)

        sentence_scores[sentence] = score

    if not sentence_scores:
        return "Error: Unable to generate summary from the provided text."

    # Determine summary length
    summary_length = min(max_sentences, max(min_sentences, len(sentences) // 3))

    # Select top-ranked sentences
    ranked_sentences = heapq.nlargest(len(sentence_scores), sentence_scores, key=sentence_scores.get)

    selected = []
    used_content = set()

    for sentence in ranked_sentences:
        sentence_key_words = set(re.findall(r'\b[a-zA-Z]+\b', sentence.lower()))

        # Simple redundancy control
        if not sentence_key_words & used_content:
            selected.append(sentence)
            used_content.update(sentence_key_words)

        if len(selected) == summary_length:
            break

    # Preserve original order
    selected.sort(key=lambda s: sentences.index(s))

    summary = " ".join(selected)

    return summary


if __name__ == "__main__":
    print("Paste your text below. Press Enter twice when finished:\n")

    user_input = ""
    while True:
        line = input()
        if line.strip() == "":
            break
        user_input += line + " "

    result = summarize_text(user_input)

    print("\nSummary:\n")
    print(result)

Paste your text below. Press Enter twice when finished:

What are AI Agents? AI agents are autonomous or semi-autonomous software systems that leverage Large Language Models (LLMs) and other AI tools to perceive information, make decisions, and perform tasks without continuous human input. Unlike traditional software programs, which operate based on pre-coded rules, AI agents can dynamically adapt their behavior based on real-time data and user intent. They can integrate APIs, access tools, generate responses, and even coordinate multi-step processes. Example Use Cases: • A personal assistant AI that manages your schedule, books meetings, and coordinates travel. • A customer service AI agent that handles inquiries, accesses internal databases, and resolves issues proactively. • An autonomous researcher that gathers sources, summarizes content, and presents reports. Defining Prompt Engineering Prompt Engineering is the art and science of crafting instructions that guide LLMs (like GPT-4